# 🧠 Brain Tumor MRI Classification with CNNs
### A Didactic Deep Learning Example for University Students
---
**Dataset:** [Brain Tumor MRI Dataset — Masoud Nickparvar (Kaggle)](https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset)

| Class | Train | Test |
|---|---|---|
| Glioma | 1 321 | 300 |
| Meningioma | 1 339 | 306 |
| No Tumor | 1 595 | 405 |
| Pituitary | 1 457 | 300 |
| **Total** | **5 712** | **1 311** |

**Learning objectives:**
1. Custom MRI preprocessing — crop black margins, resize, normalise
2. Carving a validation split from a training folder (no data leakage)
3. CNN architecture: Conv → BN → ReLU → MaxPool stacks
4. Transfer learning with ResNet-50 + two-phase fine-tuning
5. Weighted Random Sampling to handle class imbalance
6. Early stopping, cosine LR scheduling, model checkpointing
7. Evaluation: accuracy, F1, confusion matrix, ROC-AUC
8. **Grad-CAM** — visualising what the network "looks at"

## 📦 Step 0 — Install Dependencies

In [ ]:
# Run this cell once to install all required libraries
!pip install torch torchvision matplotlib scikit-learn seaborn pillow tqdm --quiet

# ── Download dataset via Kaggle CLI ──────────────────────────────────────────
# Option A (recommended if you have a kaggle.json API token):
#   !pip install kaggle --quiet
#   !kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset
#   !unzip -q brain-tumor-mri-dataset.zip -d brain-tumor-mri-dataset

# Option B — manual download:
#   Visit: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
#   Click "Download" → unzip so the folder structure looks like:
#
#   brain-tumor-mri-dataset/
#   ├── Training/
#   │   ├── glioma/        (1321 .jpg)
#   │   ├── meningioma/    (1339 .jpg)
#   │   ├── notumor/       (1595 .jpg)
#   │   └── pituitary/     (1457 .jpg)
#   └── Testing/
#       ├── glioma/        (300 .jpg)
#       ├── meningioma/    (306 .jpg)
#       ├── notumor/       (405 .jpg)
#       └── pituitary/     (300 .jpg)
print("✅ Dependencies ready")

## 📚 Step 1 — Imports

We import:
- **PyTorch** — deep learning framework (tensor ops, autograd, layers)
- **torchvision** — image datasets, transforms, pretrained models
- **scikit-learn** — evaluation metrics (F1, confusion matrix, ROC-AUC)
- **matplotlib / seaborn** — visualisation

In [ ]:
import os, sys, time, copy, random, warnings
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, transforms, models

from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve,
)
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import label_binarize
from tqdm.notebook import tqdm   # notebook-friendly progress bars

warnings.filterwarnings("ignore")
print("✅ All imports successful")

## 🔒 Step 2 — Reproducibility

> **Why?**  Deep learning involves randomness at every stage:
> weight initialisation, data shuffling, dropout masks.
> Fixing all seeds guarantees that re-running this notebook gives identical results —
> essential for debugging and comparing experiments.

In [ ]:
SEED = 42

def set_seed(seed: int) -> None:
    """Pin every source of randomness in the pipeline."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # Slightly slower, fully reproducible
    torch.backends.cudnn.benchmark = False

set_seed(SEED)
print(f"✅ Seed set to {SEED}")

## ⚙️ Step 3 — Configuration

All hyperparameters live in one dictionary.
**Change only this cell** to run different experiments — no hunting through code.

| Hyperparameter | Value | Meaning |
|---|---|---|
| `image_size` | 224 | Standard size for ImageNet-pretrained models |
| `batch_size` | 32 | Images per gradient-update step |
| `num_epochs` | 30 | Maximum training passes over the data |
| `val_fraction` | 0.15 | 15% of Training/ becomes the validation set |
| `learning_rate` | 3e-4 | Initial step size for Adam optimizer |
| `weight_decay` | 1e-4 | L2 regularisation coefficient |
| `dropout` | 0.5 | Fraction of neurons silenced during training |
| `warmup_epochs` | 5 | Epochs with frozen backbone (head-only training) |
| `label_smoothing` | 0.1 | Softens one-hot targets → better calibration |
| `patience` | 8 | Early-stop after N epochs without improvement |

In [ ]:
CONFIG = {
    # Paths — adjust if you unzipped elsewhere
    "train_dir"          : "brain-tumor-mri-dataset/Training",
    "test_dir"           : "brain-tumor-mri-dataset/Testing",
    "checkpoint"         : "best_model.pth",

    # Image
    "image_size"         : 224,

    # Training
    "batch_size"         : 32,
    "num_epochs"         : 30,
    "val_fraction"       : 0.15,

    # Optimisation
    "learning_rate"      : 3e-4,
    "weight_decay"       : 1e-4,
    "dropout"            : 0.5,

    # Transfer learning
    "use_pretrained"     : True,
    "warmup_epochs"      : 5,
    "backbone_lr_factor" : 0.1,   # backbone LR = learning_rate × this

    # Regularisation
    "label_smoothing"    : 0.1,
    "patience"           : 8,

    # Misc
    "num_classes"        : 4,
    "num_workers"        : min(4, os.cpu_count()),
}

# These MUST match the subfolder names inside Training/ and Testing/
CLASS_NAMES = ["glioma", "meningioma", "notumor", "pituitary"]

print("✅ Configuration loaded")
for k, v in CONFIG.items():
    print(f"   {k:<22} = {v}")

## 💻 Step 4 — Device Selection (CPU vs GPU)

GPUs accelerate matrix operations by running thousands of small computations
in parallel. PyTorch seamlessly moves tensors between CPU and GPU.

`torch.device("cuda")` selects the NVIDIA GPU if available; otherwise we fall back to CPU.

In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM   : {gb:.1f} GB")
else:
    print("⚠️  No GPU found — training will be slow on CPU.")

## 🖼️ Step 5 — Custom Preprocessing: `CropBlackMargins`

**Why is this needed?**

MRI scans are placed on a black (zero-intensity) background.
Large black borders waste network capacity — the model might learn to look at
the padding rather than the brain tissue.

**Algorithm:**
1. Convert image to grayscale
2. Threshold at intensity = 10 → binary mask (brain = 1, background = 0)
3. Find bounding box of the non-zero (brain) pixels
4. Crop the original colour image to that bounding box

In [ ]:
class CropBlackMargins:
    """
    Custom torchvision-compatible transform.
    Removes near-black (< threshold intensity) borders from MRI images.
    """
    def __init__(self, threshold: int = 10):
        self.threshold = threshold

    def __call__(self, img: Image.Image) -> Image.Image:
        arr  = np.array(img.convert("L"))           # Grayscale [H, W]
        mask = arr > self.threshold                  # Brain pixels = True

        if not mask.any():
            return img  # Safety: fully dark image, return unchanged

        rows = np.any(mask, axis=1)
        cols = np.any(mask, axis=0)
        rmin, rmax = np.where(rows)[0][[0, -1]]
        cmin, cmax = np.where(cols)[0][[0, -1]]

        # PIL crop(left, upper, right, lower)
        return img.crop((cmin, rmin, cmax + 1, rmax + 1))

    def __repr__(self):
        return f"CropBlackMargins(threshold={self.threshold})"

print("✅ CropBlackMargins transform defined")

## 🔄 Step 6 — Data Transforms (Augmentation & Normalisation)

**Augmentation is applied ONLY to the training set.**

| Transform | Reason |
|---|---|
| `RandomCrop` | Simulates slight position variation of the brain in the scanner |
| `RandomHorizontalFlip` | Mirror of a brain scan is still anatomically valid |
| `RandomRotation(15°)` | Scanners can position patients at slight angles |
| `ColorJitter` | Simulates brightness/contrast variation across MRI machines |
| `RandomErasing` | Randomly masks a patch — teaches robustness to occlusion |

**Normalisation** subtracts the ImageNet channel means and divides by std.
Required because we start from ImageNet-pretrained weights.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

TRANSFORMS = {
    "train": transforms.Compose([
        CropBlackMargins(threshold=10),
        transforms.Resize((CONFIG["image_size"] + 16, CONFIG["image_size"] + 16)),
        transforms.RandomCrop(CONFIG["image_size"]),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.RandomRotation(degrees=15),
        transforms.ColorJitter(brightness=0.25, contrast=0.25, saturation=0.1),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        transforms.RandomErasing(p=0.2, scale=(0.02, 0.1)),
    ]),
    "val": transforms.Compose([
        CropBlackMargins(threshold=10),
        transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
    "test": transforms.Compose([
        CropBlackMargins(threshold=10),
        transforms.Resize((CONFIG["image_size"], CONFIG["image_size"])),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ]),
}

print("✅ Transforms defined")
print("\nTRAIN pipeline:")
for t in TRANSFORMS["train"].transforms:
    print(f"   {t}")

## 📂 Step 7 — Dataset Loading & Validation Split

The Kaggle dataset provides `Training/` and `Testing/` but **no validation folder**.

**Strategy:**
- Load all of `Training/` with `ImageFolder`
- Use **`StratifiedShuffleSplit`** to split 85% / 15%
  - *Stratified* = every class keeps its proportion in both halves
  - This prevents a val set with very few meningioma images

**Data-leakage prevention:**  
`TransformSubset` wraps each split separately so training augmentation
is **never applied to validation images**.

In [ ]:
class TransformSubset(torch.utils.data.Dataset):
    """
    Wraps a Subset and applies a specific transform.
    Prevents augmentation leaking into the validation split.
    """
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform
        self.classes   = subset.dataset.classes
        self.targets   = [subset.dataset.targets[i] for i in subset.indices]

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        img, label = self.subset[idx]   # img is PIL here (no transform on parent)
        if self.transform:
            img = self.transform(img)
        return img, label


def load_data(config):
    train_path = Path(config["train_dir"])
    test_path  = Path(config["test_dir"])

    if not train_path.exists():
        print(f"❌ Training folder not found: {train_path}")
        print("   Running DEMO mode with random tensors...")
        return _demo_loaders(config)

    # Load full Training/ folder — NO transform yet (will apply per subset)
    full_train = datasets.ImageFolder(root=str(train_path), transform=None)
    # ImageFolder assigns alphabetical labels:
    # glioma=0, meningioma=1, notumor=2, pituitary=3
    print(f"   ImageFolder classes: {full_train.class_to_idx}")

    # Stratified 85/15 split
    sss = StratifiedShuffleSplit(n_splits=1, test_size=config["val_fraction"],
                                 random_state=SEED)
    train_idx, val_idx = next(sss.split(np.zeros(len(full_train.targets)),
                                        full_train.targets))

    train_ds = TransformSubset(Subset(full_train, train_idx), TRANSFORMS["train"])
    val_ds   = TransformSubset(Subset(full_train, val_idx),   TRANSFORMS["val"])
    test_ds  = datasets.ImageFolder(root=str(test_path), transform=TRANSFORMS["test"])

    datasets_dict = {"train": train_ds, "val": val_ds, "test": test_ds}

    # Weighted Random Sampler — corrects for minor class imbalance
    train_labels  = train_ds.targets
    class_counts  = np.bincount(train_labels)
    sample_w      = [1.0 / class_counts[t] for t in train_labels]
    sampler = WeightedRandomSampler(torch.DoubleTensor(sample_w),
                                    len(sample_w), replacement=True)

    loaders = {
        "train": DataLoader(train_ds, batch_size=config["batch_size"],
                            sampler=sampler, num_workers=config["num_workers"],
                            pin_memory=(DEVICE.type=="cuda")),
        "val"  : DataLoader(val_ds,   batch_size=config["batch_size"],
                            shuffle=False, num_workers=config["num_workers"],
                            pin_memory=(DEVICE.type=="cuda")),
        "test" : DataLoader(test_ds,  batch_size=config["batch_size"],
                            shuffle=False, num_workers=config["num_workers"],
                            pin_memory=(DEVICE.type=="cuda")),
    }
    sizes = {s: len(d) for s, d in datasets_dict.items()}
    return datasets_dict, loaders, sizes


def _demo_loaders(config):
    from torch.utils.data import TensorDataset
    n = 128
    X = torch.randn(n, 3, config["image_size"], config["image_size"])
    Y = torch.randint(0, config["num_classes"], (n,))
    ds = TensorDataset(X, Y)
    ld = DataLoader(ds, batch_size=config["batch_size"], shuffle=True)
    class FDS:
        targets = Y.tolist()
        def __len__(self): return n
        def __getitem__(self, i): return X[i], Y[i]
    fake = FDS()
    return ({"train":fake,"val":fake,"test":fake},
            {"train":ld,"val":ld,"test":ld},
            {"train":n,"val":n,"test":n})


print("✅ Data loading functions defined")

In [ ]:
datasets_dict, loaders, sizes = load_data(CONFIG)

print(f"\n{'Split':<8} {'N':>5}  Class distribution")
print("─" * 55)
for split, ds in datasets_dict.items():
    counts = np.bincount(ds.targets, minlength=CONFIG["num_classes"])
    bar = "  ".join(f"{CLASS_NAMES[i]}:{counts[i]}" for i in range(len(counts)))
    print(f"{split:<8} {len(ds):>5}  {bar}")

## 🔍 Step 8 — Visualise Sample Images

Let's inspect some raw training images and see the effect of `CropBlackMargins`.

In [ ]:
def show_samples(loader, n_cols=4, n_rows=2):
    """Display a grid of sample images from the loader."""
    mean = torch.tensor(IMAGENET_MEAN).view(3,1,1)
    std  = torch.tensor(IMAGENET_STD).view(3,1,1)

    imgs, labels = next(iter(loader))
    n = n_cols * n_rows

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 3, n_rows * 3))
    fig.suptitle("Sample Training Images (after augmentation)", fontsize=13,
                 fontweight="bold")
    for i, ax in enumerate(axes.flat):
        if i >= n: break
        img = (imgs[i] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
        ax.imshow(img)
        ax.set_title(CLASS_NAMES[labels[i].item()], fontsize=10, fontweight="bold")
        ax.axis("off")
    plt.tight_layout()
    plt.show()

show_samples(loaders["train"])

In [ ]:
# Class distribution bar chart
counts_train = np.bincount(datasets_dict["train"].targets)
counts_test  = np.bincount(datasets_dict["test"].targets)

x = np.arange(len(CLASS_NAMES))
width = 0.35

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(x - width/2, counts_train, width, label="Train", color="#4C72B0")
ax.bar(x + width/2, counts_test,  width, label="Test",  color="#DD8452")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES)
ax.set_ylabel("Number of images"); ax.set_title("Class Distribution")
ax.legend(); ax.grid(axis="y", alpha=0.3)
plt.tight_layout(); plt.show()

## 🏗️ Step 9 — CNN Architecture (From Scratch)

### ConvBlock: the fundamental building block

```
Conv2d → BatchNorm2d → ReLU → (MaxPool2d)
```

| Layer | What it does |
|---|---|
| **Conv2d** | Slides learnable filters over the input; each filter detects a pattern (edge, texture…) |
| **BatchNorm2d** | Normalises activations → stabilises training, acts as mild regulariser |
| **ReLU** | Non-linearity: f(x) = max(0, x). Without it, deep networks collapse to a single linear map |
| **MaxPool2d** | Downsamples 2×: keeps the strongest activation in each 2×2 window → translation invariance |

### Shape trace through BrainTumorCNN

```
Input        : [B,   3, 224, 224]
After Block1 : [B,  32, 112, 112]
After Block2 : [B,  64,  56,  56]
After Block3 : [B, 128,  28,  28]
After Block4 : [B, 256,  14,  14]
After Block5 : [B, 512,   7,   7]
After GAP    : [B, 512,   1,   1]
After Flatten: [B, 512]
Output logits: [B,   4]
```

In [ ]:
class ConvBlock(nn.Module):
    """Conv2d → BatchNorm2d → ReLU → (MaxPool2d)"""
    def __init__(self, in_ch, out_ch, pool=True):
        super().__init__()
        layers = [
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            # bias=False: BatchNorm's β parameter already provides the shift
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),   # inplace saves memory
        ]
        if pool:
            layers.append(nn.MaxPool2d(kernel_size=2, stride=2))
        self.block = nn.Sequential(*layers)

    def forward(self, x):
        return self.block(x)


class BrainTumorCNN(nn.Module):
    """5-block custom CNN — educational, builds intuition from scratch."""
    def __init__(self, num_classes=4, dropout=0.5):
        super().__init__()
        self.backbone = nn.Sequential(
            ConvBlock(  3,  32),   # 224 → 112
            ConvBlock( 32,  64),   # 112 →  56
            ConvBlock( 64, 128),   #  56 →  28
            ConvBlock(128, 256),   #  28 →  14
            ConvBlock(256, 512),   #  14 →   7
        )
        # Global Average Pooling: 512×7×7 → 512×1×1
        # Much cheaper than flattening (25088 → 512 features)
        self.gap  = nn.AdaptiveAvgPool2d((1, 1))
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(512, 256), nn.BatchNorm1d(256), nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(inplace=True),
            nn.Dropout(dropout * 0.5),
            nn.Linear(128, num_classes),  # Raw logits — NO softmax here
        )
        self._init_weights()

    def _init_weights(self):
        """Kaiming (He) init — keeps gradient variance stable at start."""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
            elif isinstance(m, (nn.BatchNorm2d, nn.BatchNorm1d)):
                nn.init.ones_(m.weight); nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, nonlinearity="relu")
                nn.init.zeros_(m.bias)

    def forward(self, x):
        x = self.backbone(x)   # Feature extraction
        x = self.gap(x)        # Spatial pooling
        return self.head(x)    # Classification


# Quick sanity check — feed a random batch and check output shape
_dummy = torch.randn(2, 3, 224, 224)
_m     = BrainTumorCNN()
_out   = _m(_dummy)
print(f"✅ BrainTumorCNN output shape: {_out.shape}  (expect [2, 4])")
n_params = sum(p.numel() for p in _m.parameters())
print(f"   Total parameters: {n_params:,}")

## 🔁 Step 10 — Transfer Learning with ResNet-50

### Why transfer learning?

ResNet-50 was trained on **1.2 million ImageNet photos** for 90 epochs.  
Its early layers encode universal features — edges, corners, textures —  
that transfer to MRI scans even though the domains differ.

### Two-phase training strategy

```
Phase 1 — Warm-up  (epochs 1 … warmup_epochs)
  Backbone:  FROZEN  (requires_grad = False)
  Head only: trains with normal LR
  Reason:    random head weights produce large gradients that could
             destroy the pretrained backbone if trained simultaneously.

Phase 2 — Fine-tuning  (epochs warmup_epochs+1 … end)
  Backbone:  UNFROZEN  (requires_grad = True)
  Backbone LR = head LR × 0.1  ← differential learning rates
  Allows backbone to specialise on MRI texture/structure.
```

### ResNet-50 architecture (simplified)

```
Input 224×224×3
  → Conv1 (7×7, stride 2)  → 112×112
  → MaxPool                → 56×56
  → Layer1 (3 residual blocks, 256 ch)
  → Layer2 (4 residual blocks, 512 ch)
  → Layer3 (6 residual blocks, 1024 ch)
  → Layer4 (3 residual blocks, 2048 ch)  ← Grad-CAM target
  → AdaptiveAvgPool                      → 1×1
  → fc (2048 → 1000)  ← we REPLACE this with our 4-class head
```

In [ ]:
def build_resnet50(num_classes=4, dropout=0.5):
    """ImageNet-pretrained ResNet-50 with a custom 4-class head."""
    model = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)

    # Freeze ALL backbone parameters for warm-up phase
    for p in model.parameters():
        p.requires_grad = False

    # Replace the original 1000-class linear head
    in_feats = model.fc.in_features   # 2048
    model.fc = nn.Sequential(
        nn.Linear(in_feats, 512), nn.BatchNorm1d(512), nn.ReLU(inplace=True),
        nn.Dropout(dropout),
        nn.Linear(512, 256),     nn.BatchNorm1d(256), nn.ReLU(inplace=True),
        nn.Dropout(dropout * 0.5),
        nn.Linear(256, num_classes),   # 4 logits
    )

    n_total     = sum(p.numel() for p in model.parameters())
    n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"✅ ResNet-50 loaded")
    print(f"   Total params    : {n_total:,}")
    print(f"   Trainable params: {n_trainable:,}  ({100*n_trainable/n_total:.1f}%)")
    print(f"   Warm-up epochs  : {CONFIG['warmup_epochs']}  (head only)")
    return model


def unfreeze_for_finetuning(model, lr):
    """Unfreeze backbone; return optimizer with differential LRs."""
    for p in model.parameters():
        p.requires_grad = True

    backbone_params = [p for n,p in model.named_parameters() if "fc" not in n]
    head_params     = list(model.fc.parameters())

    optimizer = optim.AdamW([
        {"params": backbone_params,
         "lr": lr * CONFIG["backbone_lr_factor"], "weight_decay": CONFIG["weight_decay"]},
        {"params": head_params,
         "lr": lr, "weight_decay": CONFIG["weight_decay"]},
    ])
    n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n🔓 Fine-tuning started — Trainable params: {n_train:,}")
    return optimizer

## 📉 Step 11 — Loss Function: Cross-Entropy with Label Smoothing

**Cross-Entropy Loss** measures how wrong our probability distribution is
compared to the true one-hot distribution.

$$L = -\log\left(\frac{e^{z_c}}{\sum_k e^{z_k}}\right)$$

where $z_c$ is the logit for the correct class $c$.

**Label smoothing** (ε = 0.1) replaces the hard target `[0, 0, 1, 0]` with  
`[0.033, 0.033, 0.9, 0.033]`, preventing the model from becoming over-confident
and improving probability calibration.

In [ ]:
criterion = nn.CrossEntropyLoss(label_smoothing=CONFIG["label_smoothing"])
print(f"✅ CrossEntropyLoss with label_smoothing={CONFIG['label_smoothing']}")

## 🔄 Step 12 — Training & Evaluation Loops

### The 6-step training loop (per mini-batch)

```
① zero_grad()    — clear accumulated gradients from previous step
② forward()      — compute predictions (logits)
③ loss()         — cross-entropy between logits and true labels
④ backward()     — backpropagate: compute ∂L/∂W for every parameter W
⑤ clip_grad_norm — cap gradient L2-norm to 1.0 (prevents exploding gradients)
⑥ step()         — update W ← W − lr × ∂L/∂W  (AdamW variant)
```

**`model.train()` vs `model.eval()`**
- `train()`: Dropout active, BatchNorm uses batch statistics
- `eval()` : Dropout disabled, BatchNorm uses running statistics

In [ ]:
def train_one_epoch(model, loader, optimizer, device, epoch):
    model.train()
    total_loss, correct, total = 0.0, 0, 0

    bar = tqdm(loader, desc=f"Epoch {epoch:3d} [train]", leave=False)
    for imgs, labels in bar:
        imgs, labels = imgs.to(device), labels.to(device)

        optimizer.zero_grad()                                    # ① clear grads
        logits = model(imgs)                                     # ② forward
        loss   = criterion(logits, labels)                       # ③ loss
        loss.backward()                                          # ④ backprop
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)  # ⑤ clip
        optimizer.step()                                         # ⑥ update

        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
        bar.set_postfix(loss=f"{loss.item():.3f}")

    return total_loss / total, correct / total


@torch.no_grad()
def validate(model, loader, device, phase="val"):
    """No gradient computation needed — saves memory and time."""
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    for imgs, labels in tqdm(loader, desc=f"       [{phase:5s}]", leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / total, correct / total, all_preds, all_labels

print("✅ Training and validation loops defined")

## 🚀 Step 13 — Full Training Pipeline

Combines:
- **Cosine Annealing LR** — smoothly decays LR from `lr_max` to `lr_min` following a cosine curve
- **Early Stopping** — halts training if val accuracy doesn't improve for `patience` epochs
- **Model Checkpointing** — saves the best-ever weights to disk

In [ ]:
def train_model(model, loaders, config, device):
    model = model.to(device)

    optimizer = optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=config["learning_rate"], weight_decay=config["weight_decay"])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=config["num_epochs"],
        eta_min=config["learning_rate"] * 0.01)

    history     = {k: [] for k in ("train_loss","val_loss","train_acc","val_acc","lr")}
    best_acc    = 0.0
    patience_ctr= 0
    best_weights= copy.deepcopy(model.state_dict())

    print(f"{'Ep':>3}  {'TrLoss':>7}  {'TrAcc':>6}  {'VaLoss':>7}  {'VaAcc':>6}  {'LR':>8}")
    print("─" * 50)

    for epoch in range(1, config["num_epochs"] + 1):
        # ── Switch to fine-tuning after warm-up ──────────────────────────
        if epoch == config["warmup_epochs"] + 1 and config["use_pretrained"]:
            optimizer = unfreeze_for_finetuning(model, config["learning_rate"])
            scheduler = optim.lr_scheduler.CosineAnnealingLR(
                optimizer,
                T_max  = config["num_epochs"] - config["warmup_epochs"],
                eta_min= config["learning_rate"] * 0.01)

        t0 = time.time()
        tr_loss, tr_acc = train_one_epoch(model, loaders["train"], optimizer, device, epoch)
        va_loss, va_acc, _, _ = validate(model, loaders["val"], device)
        scheduler.step()
        lr_now = scheduler.get_last_lr()[0]

        for k, v in zip(("train_loss","val_loss","train_acc","val_acc","lr"),
                        (tr_loss, va_loss, tr_acc, va_acc, lr_now)):
            history[k].append(v)

        flag = " ✓" if va_acc > best_acc else ""
        print(f"{epoch:>3}  {tr_loss:>7.4f}  {tr_acc:>6.4f}  "
              f"{va_loss:>7.4f}  {va_acc:>6.4f}  {lr_now:>8.2e}"
              f"  [{time.time()-t0:.0f}s]{flag}")

        if va_acc > best_acc:
            best_acc      = va_acc
            best_weights  = copy.deepcopy(model.state_dict())
            torch.save(best_weights, config["checkpoint"])
            patience_ctr  = 0
        else:
            patience_ctr += 1
            if patience_ctr >= config["patience"]:
                print(f"\nEarly stop — no val improvement for {config['patience']} epochs.")
                break

    model.load_state_dict(best_weights)
    print(f"\n✅ Best val accuracy: {best_acc:.4f}  →  saved to {config['checkpoint']}")
    return model, history

print("✅ Training pipeline defined")

In [ ]:
# Build and train the model
# Toggle CONFIG["use_pretrained"] = False to use the custom CNN instead
model = build_resnet50(CONFIG["num_classes"], CONFIG["dropout"]) \
        if CONFIG["use_pretrained"] else \
        BrainTumorCNN(CONFIG["num_classes"], CONFIG["dropout"])

model, history = train_model(model, loaders, CONFIG, DEVICE)

## 📊 Step 14 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4))
fig.suptitle("Training History — Brain Tumor MRI CNN", fontweight="bold", fontsize=13)
ep = range(1, len(history["train_loss"]) + 1)

axes[0].plot(ep, history["train_loss"], label="Train", lw=2)
axes[0].plot(ep, history["val_loss"],   label="Val",   lw=2, ls="--")
axes[0].set(title="Cross-Entropy Loss", xlabel="Epoch", ylabel="Loss")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(ep, history["train_acc"], label="Train", lw=2)
axes[1].plot(ep, history["val_acc"],   label="Val",   lw=2, ls="--")
axes[1].set(title="Accuracy", xlabel="Epoch", ylabel="Accuracy", ylim=[0, 1.05])
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(ep, history["lr"], color="green", lw=2)
axes[2].set(title="Learning Rate (Cosine Annealing)", xlabel="Epoch",
            ylabel="LR", yscale="log")
axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()

## 📋 Step 15 — Test Set Evaluation

We evaluate on the **held-out test set** — data the model has never seen.

**Metrics:**

| Metric | Formula | Clinical relevance |
|---|---|---|
| **Precision** | TP / (TP+FP) | Avoid false tumor detections |
| **Recall** | TP / (TP+FN) | **Don't miss real tumors!** |
| **F1-Score** | 2·P·R/(P+R) | Balance of both |
| **ROC-AUC** | Area under ROC curve | 1.0=perfect, 0.5=random |

In [ ]:
@torch.no_grad()
def full_evaluate(model, loader, device):
    model.eval()
    preds, labels, probs = [], [], []

    for imgs, lbls in tqdm(loader, desc="Testing"):
        logits = model(imgs.to(device))
        p      = torch.softmax(logits, dim=1)
        preds.extend(logits.argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
        probs.extend(p.cpu().numpy())

    preds  = np.array(preds)
    labels = np.array(labels)
    probs  = np.array(probs)

    print("\n── Classification Report ──────────────────────────────────")
    print(classification_report(labels, preds, target_names=CLASS_NAMES, digits=4))

    try:
        lb  = label_binarize(labels, classes=range(CONFIG["num_classes"]))
        auc = roc_auc_score(lb, probs, multi_class="ovr", average="macro")
        print(f"Macro ROC-AUC: {auc:.4f}")
    except Exception as e:
        auc = None
        print(f"ROC-AUC skipped: {e}")

    return {"preds": preds, "labels": labels, "probs": probs, "auc": auc}


results = full_evaluate(model, loaders["test"], DEVICE)

## 🔲 Step 16 — Confusion Matrix

In [ ]:
cm   = confusion_matrix(results["labels"], results["preds"])
cm_n = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
fig.suptitle("Confusion Matrix — Brain Tumor MRI CNN", fontweight="bold")

for ax, data, title, fmt in zip(axes,
    [cm, cm_n],
    ["Raw Counts", "Normalised (Recall per class)"],
    ["d", ".3f"]):
    sns.heatmap(data, annot=True, fmt=fmt, cmap="Blues",
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                ax=ax, linewidths=0.5)
    ax.set(title=title, xlabel="Predicted", ylabel="True")

plt.tight_layout(); plt.show()

## 📈 Step 17 — ROC Curves (One-vs-Rest)

In [ ]:
lb     = label_binarize(results["labels"], classes=range(CONFIG["num_classes"]))
colors = plt.cm.Set1(np.linspace(0, 0.8, CONFIG["num_classes"]))

fig, ax = plt.subplots(figsize=(8, 6))
for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
    fpr, tpr, _ = roc_curve(lb[:, i], results["probs"][:, i])
    auc_i = roc_auc_score(lb[:, i], results["probs"][:, i])
    ax.plot(fpr, tpr, color=col, lw=2, label=f"{cls}  AUC={auc_i:.3f}")

ax.plot([0,1],[0,1],"k--",lw=1)
ax.set(xlabel="False Positive Rate", ylabel="True Positive Rate",
       title="ROC Curves (One-vs-Rest)", xlim=[0,1], ylim=[0,1.02])
ax.legend(loc="lower right"); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 🖼️ Step 18 — Sample Predictions

In [ ]:
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD).view(3,1,1)

model.eval()
fig, axes = plt.subplots(4, 4, figsize=(14, 14))
fig.suptitle("Test Predictions  (green = correct  |  red = wrong)",
             fontsize=12, fontweight="bold")
shown = 0

with torch.no_grad():
    for imgs, lbls in loaders["test"]:
        preds = model(imgs.to(DEVICE)).argmax(1).cpu()
        for img, true, pred in zip(imgs, lbls, preds):
            if shown >= 16: break
            ax  = axes[shown//4][shown%4]
            npimg = (img * std_t + mean_t).clamp(0,1).permute(1,2,0).numpy()
            ax.imshow(npimg)
            col = "green" if true == pred else "red"
            ax.set_title(f"T: {CLASS_NAMES[true]}\nP: {CLASS_NAMES[pred]}",
                         fontsize=8, color=col, fontweight="bold")
            ax.axis("off")
            shown += 1
        if shown >= 16: break

plt.tight_layout(); plt.show()

## 🔥 Step 19 — Grad-CAM: Explainability

**Gradient-weighted Class Activation Mapping** highlights which image regions
drove the model's prediction.

### Algorithm

1. Forward pass → record feature maps $A^k$ at the last conv layer
2. Backpropagate only the predicted-class score (not the full loss)
3. Importance weight: $\alpha_k = \frac{1}{Z}\sum_{ij} \frac{\partial y^c}{\partial A^k_{ij}}$
4. Heatmap: $L^c = \text{ReLU}\!\left(\sum_k \alpha_k A^k\right)$
5. Upsample to input resolution and overlay

**Medical relevance:** Confirms the network focuses on the tumor mass,
not on scanner artefacts or image borders.

In [ ]:
class GradCAM:
    """Gradient-weighted Class Activation Mapping for PyTorch models."""
    def __init__(self, model, target_layer):
        self.model = model
        self._acts = self._grads = None
        target_layer.register_forward_hook(
            lambda m,i,o: setattr(self, "_acts", o.detach()))
        target_layer.register_full_backward_hook(
            lambda m,gi,go: setattr(self, "_grads", go[0].detach()))

    def __call__(self, x, class_idx=None):
        self.model.eval()
        x      = x.unsqueeze(0).to(DEVICE)
        logits = self.model(x)
        idx    = class_idx if class_idx is not None else logits.argmax(1).item()

        self.model.zero_grad()
        logits[0, idx].backward()   # Gradient of class score only

        weights = self._grads.mean(dim=(2,3), keepdim=True)   # α_k
        cam     = (weights * self._acts).sum(dim=1).squeeze()
        cam     = torch.relu(cam)
        cam     = (cam - cam.min()) / (cam.max() + 1e-8)
        return cam.cpu().numpy()

print("✅ GradCAM class defined")

In [ ]:
# Attach Grad-CAM to the last residual block's last conv
# (layer4[-1].conv3 is the deepest feature-extractor in ResNet-50)
try:
    target_layer = model.layer4[-1].conv3     # ResNet-50
except AttributeError:
    target_layer = model.backbone[-1].block[0]  # Custom CNN fallback

cam_fn = GradCAM(model, target_layer)
mean_t = torch.tensor(IMAGENET_MEAN).view(3,1,1)
std_t  = torch.tensor(IMAGENET_STD).view(3,1,1)

n = 8
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle("Grad-CAM — Where the CNN focuses for its prediction",
             fontweight="bold", fontsize=13)
shown = 0

for imgs, lbls in loaders["test"]:
    for img, lbl in zip(imgs, lbls):
        if shown >= n: break
        cam   = cam_fn(img)
        npimg = (img * std_t + mean_t).clamp(0,1).permute(1,2,0).numpy()
        cam_up= np.array(
            Image.fromarray((cam*255).astype(np.uint8))
                 .resize((CONFIG["image_size"], CONFIG["image_size"]), Image.BILINEAR)
        ) / 255.0

        ax = axes.flat[shown]
        ax.imshow(npimg)
        ax.imshow(cam_up, cmap="jet", alpha=0.45)
        ax.set_title(CLASS_NAMES[lbl.item()], fontsize=10, fontweight="bold")
        ax.axis("off")
        shown += 1
    if shown >= n: break

plt.tight_layout(); plt.show()

## ✅ Step 20 — Summary & Key Takeaways

| # | Concept | Where used |
|---|---|---|
| 1 | `CropBlackMargins` | Dataset-specific MRI preprocessing |
| 2 | `StratifiedShuffleSplit` | Val split without data leakage |
| 3 | `WeightedRandomSampler` | Class imbalance correction |
| 4 | `ConvBlock` (Conv→BN→ReLU→MaxPool) | Core CNN building block |
| 5 | Global Average Pooling | Compact spatial representation |
| 6 | Transfer Learning (ResNet-50) | Reuse ImageNet knowledge |
| 7 | Two-phase training (warm-up → fine-tune) | Safe pretrained model adaptation |
| 8 | Differential learning rates | Backbone vs. head update sizes |
| 9 | Cosine LR Annealing | Smooth learning rate decay |
| 10 | Label Smoothing | Better-calibrated probabilities |
| 11 | Early Stopping | Automatic overfitting prevention |
| 12 | `CrossEntropyLoss` = LogSoftmax + NLLLoss | Multi-class classification loss |
| 13 | Backpropagation | Chain rule for gradient computation |
| 14 | Grad-CAM | Visual explanation of predictions |
| 15 | ROC-AUC, F1, Confusion Matrix | Clinical evaluation metrics |

---
*Expected performance with ResNet-50 + fine-tuning: **~97–98% test accuracy***